# Auditoria PNAD: escolaridade (VD3004) e conectividade (TIC 2025) — v0.6

**Apoio ao simulador de pesquisas eleitorais · Unidade 05 · 15 de outubro de 2026**

Este notebook dá continuidade ao piloto sintético de **50 mil eleitores (TSE, v0.4)**. Ele **não simula intenções de voto** e não representa os métodos de nenhum instituto de pesquisa. O objetivo é corrigir a aproximação educacional (usar `VD3004` em vez de inferir níveis a partir de `VD3005`) e acrescentar **imputações condicionais de acesso à internet e posse de celular**, usando a PNAD TIC 2025.

**Duas fontes distintas:** PNAD trimestral de 2026 (2º trimestre) e PNAD TIC de 2025 (4º trimestre). Não existe ligação pessoa a pessoa entre elas; cada módulo fornece doadores separados. O universo final é o cadastro eleitoral TSE 2026 no Brasil, e não a população residente. Os atributos de 2025 serão identificados como **imputados**, nunca observados para eleitores sintéticos de 2026.

Fontes: [PNAD IBGE](https://www.ibge.gov.br/estatisticas/sociais/populacao/17270-pnad-continua.html) · [PNAD TIC T4 2025](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Anual/Microdados/Trimestre/Trimestre_4/Dados/) · [Questionário TIC 2025](https://biblioteca.ibge.gov.br/visualizacao/instrumentos_de_coleta/doc5745.pdf).

**Tempo e recursos:** os dois downloads podem somar centenas de megabytes e a análise em R pode demorar; use um ambiente Colab com recursos suficientes. Execute as etapas em ordem.

In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile, hashlib
from IPython.display import display
import pandas as pd

BASE = Path('/content/auditoria_eleitoral_v0_6') if 'google.colab' in sys.modules else Path.cwd() / 'auditoria_eleitoral_v0_6_runtime'
BASE.mkdir(parents=True, exist_ok=True)
os.chdir(BASE)
print('Pasta de trabalho:', BASE)

## 1. Arquivos de execução e população TSE previamente verificada

Os programas estão incluídos nas próximas células e o piloto v0.4 será recuperado de uma cópia compactada incorporada ao notebook. **Não é preciso baixar outro ZIP nem consultar um repositório GitHub.** A versão v0.4 preserva as distribuições cadastrais do TSE, sem preferências eleitorais.

In [ ]:
from pathlib import Path
Path('gerar_piloto_v06.py').write_text('#!/usr/bin/env python3\n"""Gera piloto socioeconômico + TIC, SOMENTE com extrações oficiais verificadas.\n\nEntradas:\n- piloto TSE 50k v0.4 (preservado sem alterações)\n- PNAD trimestral 2026/T2 reextraída com VD3004 e peso de survey\n- PNAD TIC 2025/T4 com S07001, S07006 e peso de survey\n\nNÃO confundir atributos TIC 2025 imputados com acesso individual observado em 2026.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nUF_COD={11:\'RO\',12:\'AC\',13:\'AM\',14:\'RR\',15:\'PA\',16:\'AP\',17:\'TO\',21:\'MA\',22:\'PI\',23:\'CE\',24:\'RN\',25:\'PB\',26:\'PE\',27:\'AL\',28:\'SE\',29:\'BA\',31:\'MG\',32:\'ES\',33:\'RJ\',35:\'SP\',41:\'PR\',42:\'SC\',43:\'RS\',50:\'MS\',51:\'MT\',52:\'GO\',53:\'DF\'}\nREGIAO={**{u:\'N\' for u in [\'RO\',\'AC\',\'AM\',\'RR\',\'PA\',\'AP\',\'TO\']},**{u:\'NE\' for u in [\'MA\',\'PI\',\'CE\',\'RN\',\'PB\',\'PE\',\'AL\',\'SE\',\'BA\']},**{u:\'SE\' for u in [\'MG\',\'ES\',\'RJ\',\'SP\']},**{u:\'S\' for u in [\'PR\',\'SC\',\'RS\']},**{u:\'CO\' for u in [\'MS\',\'MT\',\'GO\',\'DF\']}}\n# Não equivale a alfabetização; categorias 1 e 2 incluem pessoas que não são distinguíveis na VD3004.\nTSE_PARA_VD3004={\n    \'ANALFABETO\':1,\n    \'LÊ E ESCREVE\':1,\n    \'ENSINO FUNDAMENTAL INCOMPLETO\':2,\n    \'ENSINO FUNDAMENTAL COMPLETO\':3,\n    \'ENSINO MÉDIO INCOMPLETO\':4,\n    \'ENSINO MÉDIO COMPLETO\':5,\n    \'SUPERIOR INCOMPLETO\':6,\n    \'SUPERIOR COMPLETO\':7,\n}\nVD3004_ROTULOS={1:\'Sem instrução e menos de 1 ano\',2:\'Fundamental incompleto\',3:\'Fundamental completo\',4:\'Médio incompleto\',5:\'Médio completo\',6:\'Superior incompleto\',7:\'Superior completo\'}\nAMBIGUAS={\'ANALFABETO\',\'LÊ E ESCREVE\'}\nSEED=20261017\n\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for chunk in iter(lambda:f.read(1024*1024),b\'\'):h.update(chunk)\n    return h.hexdigest()\n\n\ndef faixa(s):\n    if s==\'100 anos ou mais\':return (100,130)\n    nums=[int(x) for x in re.findall(r\'\\d+\',str(s))]\n    if not nums:raise ValueError(\'Faixa não reconhecida: \'+str(s))\n    return (nums[0],nums[-1])\n\n\ndef validar_csv(df,campos,origem):\n    faltam=set(campos)-set(df.columns)\n    if faltam:raise ValueError(f\'{origem}: faltam colunas {sorted(faltam)}; REEXTRAIA em vez de imputar códigos inexistentes\')\n\n\ndef preparacao_base(df,piloto,nome,cols_obs,controle_peso):\n    validar_csv(df,[\'UF\',\'V1022\',\'V2007\',\'V2009\',\'VD3004\',controle_peso]+cols_obs,nome)\n    df=df.copy()\n    for campo in [\'UF\',\'V1022\',\'V2007\',\'V2009\',\'VD3004\']+cols_obs:\n        df[campo]=pd.to_numeric(df[campo],errors=\'coerce\')\n    df[controle_peso]=pd.to_numeric(df[controle_peso],errors=\'coerce\')\n    if df[controle_peso].isna().any() or (df[controle_peso]<=0).any():\n        raise ValueError(nome+\': pesos faltantes, zero ou negativos\')\n    if not df.UF.isin(UF_COD).all() or not df.V1022.isin([1,2]).all() or not df.V2007.isin([1,2]).all():\n        raise ValueError(nome+\': códigos UF/V1022/V2007 inválidos\')\n    if not df.V2009.between(16,130).all():raise ValueError(nome+\': idade inválida (<16 ou >130)\')\n    # VD3004 pode ser N/A pontual; esses registros não são usados como doadores comparáveis.\n    df=df[df.VD3004.isin(range(1,8))].copy()\n    if df.empty:raise ValueError(nome+\': nenhum doador com VD3004 válido\')\n    df[\'uf\']=df.UF.map(UF_COD)\n    df[\'regiao\']=df.uf.map(REGIAO)\n    df[\'sexo\']=df.V2007.map({1:\'Homem\',2:\'Mulher\'})\n    df[\'urbana\']=df.V1022.eq(1).astype(\'int8\')\n    df[\'educ\']=df.VD3004.astype(\'int8\')\n    faixas=list(piloto.faixa_etaria_tse.unique())\n    df[\'faixa_etaria_tse\']=\'\'\n    for f in faixas:\n        lo,hi=faixa(f)\n        df.loc[df.V2009.between(lo,hi),\'faixa_etaria_tse\']=f\n    if df.faixa_etaria_tse.eq(\'\').any():raise ValueError(nome+\': TSE não cobre todas as idades PNAD\')\n    return df.reset_index(drop=True)\n\n\ndef indice(df,cols):\n    return {k if isinstance(k,tuple) else (k,):np.array(ind,dtype=\'int32\') for k,ind in df.groupby(cols,dropna=False,sort=True).indices.items()}\n\n\ndef amostrar(rng,pools,weights,keys,columns,target,campos,min_pool=1):\n    """Amostra ponderada por estrato com hierarquia explícita e flags de fallback."""\n    ids=np.full(len(target),-1,dtype=\'int32\')\n    rotulos=np.full(len(target),\'\',dtype=object)\n    poolsizes=np.zeros(len(target),dtype=\'int32\')\n    for prior,campos_indice in enumerate(columns):\n        idx=indice(pools,campos_indice)\n        pendentes=np.flatnonzero(ids<0)\n        if not len(pendentes):break\n        for key,pos in target.iloc[pendentes].groupby(campos_indice,sort=True,dropna=False).indices.items():\n            key=key if isinstance(key,tuple) else (key,)\n            pool=idx.get(key)\n            if pool is None or len(pool)<min_pool:continue\n            pos=pendentes[np.asarray(pos,dtype=\'int32\')]\n            p=weights[pool].copy();p/=p.sum()\n            ids[pos]=rng.choice(pool,size=len(pos),replace=True,p=p)\n            rotulos[pos]=f\'NIVEL_{prior}_\'+(\'_\'.join(campos_indice))\n            poolsizes[pos]=len(pool)\n    if (ids<0).any():\n        faltas=target.iloc[np.flatnonzero(ids<0)][campos].drop_duplicates().head(10)\n        raise ValueError(f\'Sem doador {int((ids<0).sum())} registros:\\n{faltas}\')\n    return ids,rotulos,poolsizes\n\n\ndef executar(piloto_path,base2026_path,tic_path,saida,seed=SEED):\n    saida=Path(saida);saida.mkdir(parents=True,exist_ok=True)\n    p=pd.read_csv(piloto_path)\n    validar_csv(p,[\'eleitor_id\',\'uf\',\'sexo_comum\',\'faixa_etaria_tse\',\'escolaridade_tse\'],\'TSE\')\n    if not p.eleitor_id.is_unique:raise ValueError(\'IDs TSE repetidos\')\n    if len(p)!=50000:raise ValueError(\'Esperados 50000 registros TSE v0.4\')\n    if not p.escolaridade_tse.isin(TSE_PARA_VD3004).all():\n        raise ValueError(\'Categoria escolaridade TSE desconhecida\')\n    if not p.uf.isin(REGIAO).all():raise ValueError(\'UF não reconhecida\')\n    p[\'regiao\']=p.uf.map(REGIAO)\n    p[\'educ\']=p.escolaridade_tse.map(TSE_PARA_VD3004).astype(\'int8\')\n    p[\'educacao_ambigua\']=p.escolaridade_tse.isin(AMBIGUAS)\n\n    cols2026=[\'UF\',\'V1022\',\'V2007\',\'V2009\',\'VD3004\',\'VD3005\',\'VD4001\',\'VD4002\',\'VD4020\',\'peso_pnad\']\n    cols_tic=[\'UF\',\'V1022\',\'V2007\',\'V2009\',\'VD3004\',\'S07001\',\'S07006\',\'peso_tic\']\n    d=pd.read_csv(base2026_path,compression=\'infer\')\n    t=pd.read_csv(tic_path,compression=\'infer\')\n    validar_csv(d,cols2026,\'PNAD 2026T2\')\n    validar_csv(t,cols_tic,\'PNAD TIC 2025T4\')\n    if not d.V2009.ge(16).all() or not t.V2009.ge(16).all():\n        raise ValueError(\'Arquivos devem conter somente moradores 16+\')\n    d=preparacao_base(d,p,\'PNAD trimestral 2026\', [\'VD3005\',\'VD4001\',\'VD4002\',\'VD4020\'],\'peso_pnad\')\n    t=preparacao_base(t,p,\'PNAD TIC 2025\',[\'S07001\',\'S07006\'],\'peso_tic\')\n    # Filtragem de S07001 e S07006 válida. 1 sim / 2 não; NAs não são convertidos em \'não\'.\n    t=t.loc[t.S07001.isin([1,2]) & t.S07006.isin([1,2])].reset_index(drop=True)\n    if t.empty:raise ValueError(\'Nenhum registro TIC 2025 com S07001/S07006 observados\')\n    rng=np.random.default_rng(seed)\n    # NIVEL_0: UF + sexo + faixa exata + grau de instrução; NIVEL_1 reduz instrução;\n    # NIVEL_2 reduz sexo; NIVEL_3 reduz UF mas mantém região; NIVEL_4 somente faixa;\n    # NIVEL_5 nacional. Um estrato vazio NUNCA é tratado como completo.\n    chave_2026=[\n       [\'uf\',\'sexo\',\'faixa_etaria_tse\',\'educ\'],\n       [\'uf\',\'sexo\',\'faixa_etaria_tse\'],\n       [\'uf\',\'faixa_etaria_tse\'],\n       [\'regiao\',\'faixa_etaria_tse\'],\n       [\'faixa_etaria_tse\'],\n       [\'regiao\'],\n    ]\n    alvo=p.rename(columns={\'sexo_comum\':\'sexo\'}).copy()\n    pesos_d=d.peso_pnad.to_numpy(dtype=float)\n    idx,metodo,n_eleg=amostrar(rng,d,pesos_d,None,chave_2026,alvo,[\'uf\',\'sexo\',\'faixa_etaria_tse\',\'educ\'],min_pool=1)\n    do=d.iloc[idx].reset_index(drop=True)\n    out=p.drop(columns=[\'regiao\',\'educ\']).copy()\n    # original 8 colunas ficam inalteradas na mesma ordem\n    out[\'idade_pnad_doador\']=do.V2009.to_numpy(dtype=int)\n    out[\'sexo_pnad_doador\']=do.sexo.to_numpy()\n    out[\'nivel_instrucao_vd3004\']=do.educ.to_numpy(dtype=int)\n    out[\'anos_estudo_pnad_doador\']=do.VD3005.to_numpy()\n    out[\'situacao_domicilio_pnad\']=np.where(do.urbana.eq(1),\'Urbana\',\'Rural\')\n    out[\'situacao_forca_trabalho\']=do.VD4001.map({1:\'Na força de trabalho\',2:\'Fora da força de trabalho\'}).to_numpy()\n    out[\'situacao_ocupacao\']=do.VD4002.map({1.:\'Ocupada\',2.:\'Desocupada\'}).fillna(\'Fora da força de trabalho\').to_numpy()\n    out[\'rendimento_efetivo_trabalho_pnad\']=do.VD4020.to_numpy()\n    out[\'doador_peso_pnad_original\']=do.peso_pnad.to_numpy()\n    out[\'metodo_pareamento_2026\']=metodo\n    out[\'n_doadores_elegiveis_2026\']=n_eleg\n    out[\'instrucao_vd3004_coincide\']=do.educ.to_numpy()==p.educ.to_numpy()\n    # ALERTA: VD3004=1 não permite identificar ser ou não alfabetizado; NÃO chamar de correspondência exata.\n    out[\'instrucao_tse_sem_correspondencia_alfabetizacao\']=p.educacao_ambigua.to_numpy()\n    out[\'regiao\']=p.regiao.to_numpy()\n    out[\'urbana\']=do.urbana.to_numpy()\n    out[\'educ\']=do.educ.to_numpy()\n\n    alvo_tic=out.rename(columns={\'sexo_comum\':\'sexo\'}).copy()\n    cols_tic_index=[\n       [\'uf\',\'sexo\',\'faixa_etaria_tse\',\'educ\',\'urbana\'],\n       [\'uf\',\'sexo\',\'faixa_etaria_tse\',\'urbana\'],\n       [\'uf\',\'faixa_etaria_tse\',\'urbana\'],\n       [\'regiao\',\'sexo\',\'faixa_etaria_tse\',\'urbana\'],\n       [\'regiao\',\'faixa_etaria_tse\',\'urbana\'],\n       [\'regiao\',\'faixa_etaria_tse\'],\n       [\'faixa_etaria_tse\'],\n       [\'regiao\'],\n    ]\n    ids_tic,metodos_tic,n_tic=amostrar(rng,t,t.peso_tic.to_numpy(dtype=float),None,cols_tic_index,alvo_tic,[\'uf\',\'sexo\',\'faixa_etaria_tse\',\'educ\',\'urbana\'],min_pool=10)\n    donor_tic=t.iloc[ids_tic].reset_index(drop=True)\n    out[\'internet_uso_3m_pnad_tic_2025_imputado\']=(donor_tic.S07001.to_numpy()==1).astype(\'int8\')\n    out[\'celular_pessoal_pnad_tic_2025_imputado\']=(donor_tic.S07006.to_numpy()==1).astype(\'int8\')\n    out[\'metodo_imputacao_tic\']=metodos_tic\n    out[\'n_doadores_elegiveis_tic\']=n_tic\n    out=out.drop(columns=[\'regiao\',\'urbana\',\'educ\',\'educacao_ambigua\'])\n    if not out.eleitor_id.equals(p.eleitor_id):raise AssertionError(\'IDs não foram preservados\')\n    for col in [\'uf\',\'sexo_registrado\',\'faixa_etaria_tse\',\'escolaridade_tse\']:\n        if not out[col].equals(p[col]):raise AssertionError(\'TSE mudou \'+col)\n    if not out.internet_uso_3m_pnad_tic_2025_imputado.isin([0,1]).all():raise AssertionError(\'Internet inválida\')\n    if not out.celular_pessoal_pnad_tic_2025_imputado.isin([0,1]).all():raise AssertionError(\'Celular inválido\')\n    path=saida/\'piloto_socioeconomico_tic_50mil_v0_6.csv.gz\'\n    out.to_csv(path,index=False,compression={\'method\':\'gzip\',\'compresslevel\':6,\'mtime\':0})\n    # Controle separado: frequências ponderadas diretamente na PNAD TIC 2025 (16+),\n    # distinto dos percentuais imputados no piloto TSE 2026.\n    diagnostico=[]\n    for uf,grupo in [(\'BR\',t)]+list(t.groupby(\'uf\',sort=True)):\n        w=grupo.peso_tic.to_numpy(dtype=float)\n        diagnostico.append({\n            \'uf\':uf,\'n_registros_tic\':len(grupo),\n            \'massa_peso_tic\':float(w.sum()),\n            \'uso_internet_3m_ponderado_pct\':float(100*np.average(grupo.S07001.eq(1),weights=w)),\n            \'posse_celular_ponderado_pct\':float(100*np.average(grupo.S07006.eq(1),weights=w))\n        })\n    pd.DataFrame(diagnostico).to_csv(saida/\'diagnostico_tic_fonte_ponderada_v0_6.csv\',index=False)\n    meta={\n        \'status\':\'PILOTO_PROVISORIO_NAO_HOMOLOGADO\',\n        \'n_eleitores\':len(out),\'n_pnad_trimestral_validos\':len(d),\'n_tic_validos\':len(t),\'semente\':int(seed),\n        \'sha256_entradas\':{str(Path(x).name):sha256(x) for x in [piloto_path,base2026_path,tic_path]},\n        \'sha256_piloto_saida\':sha256(path),\n        \'pareamento_2026\':pd.Series(metodo).value_counts().to_dict(),\n        \'pareamento_tic\':pd.Series(metodos_tic).value_counts().to_dict(),\n        \'ambiguidade_alfabetizacao_tse\':int(out.instrucao_tse_sem_correspondencia_alfabetizacao.sum()),\n        \'concordancia_vd3004_apenas_categorias_comparaveis_pct\':round(float(100*out.loc[~out.instrucao_tse_sem_correspondencia_alfabetizacao,\'instrucao_vd3004_coincide\'].mean()),3),\n        \'uso_internet_3m_pct_imputado\':round(float(100*out.internet_uso_3m_pnad_tic_2025_imputado.mean()),3),\n        \'posse_celular_pct_imputado\':round(float(100*out.celular_pessoal_pnad_tic_2025_imputado.mean()),3),\n        \'avisos\':[\'TSE cadastro eleitores e PNAD população residente são universos distintos\',\n                  \'Padrões TIC 2025 foram imputados aos perfis TSE 2026 por estratos, NÃO medidos nesses eleitores\',\n                  \'S07001: usou Internet nos últimos três meses, NÃO equivale a estar elegível ou exposto a convite digital\',\n                  \'S07006: posse celular pessoal, NÃO garante Internet pelo aparelho\',\n                  \'TSE analfabeto/lê e escreve não são separados com VD3004 (nível 1)\',\n                  \'VD3004 nível de instrução não é idêntico à escolaridade no cadastro TSE\',\n                  \'Não calibrado por renda ou ocupação e não contém opinião eleitoral\']\n    }\n    (saida/\'manifesto_v0_6.json\').write_text(json.dumps(meta,indent=2,ensure_ascii=False)+\'\\n\',encoding=\'utf-8\')\n    display=out.groupby(\'escolaridade_tse\',dropna=False).agg(n=(\'eleitor_id\',\'size\'),pct_internet_imputada=(\'internet_uso_3m_pnad_tic_2025_imputado\',lambda x:round(100*x.mean(),3)),n_ambigua=(\'instrucao_tse_sem_correspondencia_alfabetizacao\',\'sum\'))\n    display.to_csv(saida/\'diagnostico_escolaridade_digital_v0_6.csv\')\n    print(json.dumps(meta,ensure_ascii=False,indent=2))\n    return meta\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--piloto\',required=True,type=Path)\n    parser.add_argument(\'--pnad2026\',required=True,type=Path)\n    parser.add_argument(\'--tic2025\',required=True,type=Path)\n    parser.add_argument(\'--saida\',required=True,type=Path)\n    parser.add_argument(\'--semente\',type=int,default=SEED)\n    a=parser.parse_args()\n    executar(a.piloto,a.pnad2026,a.tic2025,a.saida,a.semente)\n', encoding='utf-8')
print('Programa preparado:', 'gerar_piloto_v06.py')

In [ ]:
from pathlib import Path
Path('reextrair_pnad_2026T2_vd3004.R').write_text("# Reextração OFICIAL da PNAD-C trimestral 2026 T2 com VD3004.\n# Executar em R + Internet, não necessita nem altera os doadores v0.5.\nif (!requireNamespace('PNADcIBGE',quietly=TRUE)) stop('Instale PNADcIBGE')\nif (!requireNamespace('survey',quietly=TRUE)) stop('Instale survey')\nout <- 'saida_fontes_v0_6'\ndir.create(out,recursive=TRUE,showWarnings=FALSE)\ndir.create(file.path(out,'downloads_trimestral'),recursive=TRUE,showWarnings=FALSE)\nvars <- c('UF','V1022','V2007','V2009','VD3004','VD3005','VD4001','VD4002','VD4020')\ncat('Carregando PNAD-C trimestral, 2026 2º trimestre. Este download é volumoso.\\n')\ndes <- PNADcIBGE::get_pnadc(\n  year=2026, quarter=2, vars=vars, labels=FALSE,deflator=FALSE,\n  design=TRUE,reload=FALSE,savedir=file.path(out,'downloads_trimestral')\n)\nif (is.null(des)) stop('Falha ao carregar PNAD 2026T2')\nbase <- des$variables\nfaltantes <- setdiff(vars,names(base))\nif (length(faltantes)>0) stop(paste('Variáveis não disponíveis:',paste(faltantes,collapse=', ')))\n# Restringimos a 16+ sem tornar o total expandido igual ao do TSE.\ndes <- subset(des, !is.na(V2009) & as.numeric(as.character(V2009))>=16)\nbase <- des$variables\nbase$peso_pnad <- as.numeric(survey::weights(des,type='sampling'))\nif (anyNA(base$peso_pnad) || any(base$peso_pnad<=0)) stop('Pesos inválidos')\n# Guardar SOMENTE campos usados; linhas com VD3004 ausente seguem no arquivo para auditoria.\nfn <- file.path(out,'pnad_doadores_vd3004_2026T2.csv.gz')\ncon <- gzfile(fn,open='wt')\nwrite.csv(base[,c(vars,'peso_pnad'),drop=FALSE],con,row.names=FALSE,na='')\nclose(con)\ncat('Registros com 16+: ',nrow(base),'\\n')\ncat('Códigos VD3004 (incl. NA):\\n')\nprint(table(base$VD3004,useNA='ifany'))\ncat('Arquivo gravado: ',fn,'\\n')\n", encoding="utf-8")
print("Programa preparado:", 'reextrair_pnad_2026T2_vd3004.R')

In [ ]:
from pathlib import Path
Path('extrair_tic_2025T4.R').write_text("# Extração OFICIAL da PNAD-C TIC, acumulada no 4º trimestre de 2025.\n# O módulo TIC descreve residentes 10+, mas aqui filtramos para 16+.\n# S07001: utilização pessoal da internet NOS ÚLTIMOS 3 MESES.\n# S07006: posse pessoal de telefone móvel celular.\n# Ambos 1=sim 2=não (códigos da documentação da PNAD).\nif (!requireNamespace('PNADcIBGE',quietly=TRUE)) stop('Instale PNADcIBGE')\nif (!requireNamespace('survey',quietly=TRUE)) stop('Instale survey')\nout <- 'saida_fontes_v0_6'\ndir.create(out,recursive=TRUE,showWarnings=FALSE)\ndir.create(file.path(out,'downloads_tic'),recursive=TRUE,showWarnings=FALSE)\nvars <- c('UF','V1022','V2007','V2009','VD3004','S07001','S07006')\ncat('Carregando PNAD-C TIC 2025, módulo trimestral 4. Este download é volumoso.\\n')\ndes <- PNADcIBGE::get_pnadc(\n  year=2025,topic=4,vars=vars,labels=FALSE,deflator=FALSE,\n  design=TRUE,reload=FALSE,savedir=file.path(out,'downloads_tic')\n)\nif (is.null(des)) stop('Falha ao carregar PNAD TIC 2025T4')\nbase <- des$variables\nfaltantes <- setdiff(vars,names(base))\nif (length(faltantes)>0) stop(paste('Variáveis TIC ausentes:',paste(faltantes,collapse=', ')))\ndes <- subset(des, !is.na(V2009) & as.numeric(as.character(V2009))>=16)\nbase <- des$variables\nbase$peso_tic <- as.numeric(survey::weights(des,type='sampling'))\nif (anyNA(base$peso_tic) || any(base$peso_tic<=0)) stop('Pesos TIC inválidos')\n# Preservar NAs e verificar códigos; não recodificar ausência como zero/não.\nfor(v in c('S07001','S07006')){\n  cat('Frequência ',v,' (NA inclusive):\\n');print(table(base[[v]],useNA='ifany'))\n  z <- unique(as.character(base[[v]][!is.na(base[[v]])]))\n  if(!all(z %in% c('1','2'))) stop(paste('Código inesperado em',v,':',paste(z,collapse=',')))\n}\nfn <- file.path(out,'pnad_tic_2025T4_16mais.csv.gz')\ncon <- gzfile(fn,open='wt')\nwrite.csv(base[,c(vars,'peso_tic'),drop=FALSE],con,row.names=FALSE,na='')\nclose(con)\ncat('Registros TIC 16+: ',nrow(base),'\\n')\ncat('Arquivo gravado: ',fn,'\\n')\n", encoding="utf-8")
print("Programa preparado:", 'extrair_tic_2025T4.R')

In [ ]:
import base64, zlib
from pathlib import Path
Path('piloto_demografico_tse_50mil.csv').write_bytes(zlib.decompress(base64.b64decode('...')))
p=pd.read_csv('piloto_demografico_tse_50mil.csv')
assert len(p)==50000 and p.eleitor_id.is_unique
print('Piloto de referência carregado:',len(p),'eleitores sintéticos')
display(p.head())

## 2. Preparar R e obter as fontes oficiais

A execução exige instalação de R, `PNADcIBGE` e `survey`. Só execute esta célula se puder aguardar os downloads e instalações. Se houver falha por conectividade ou mudança de estrutura no servidor IBGE, **não substitua as tabelas por números hipotéticos**: compartilhe a mensagem de erro para corrigirmos o extrator.

Os dois arquivos oficiais serão obtidos novamente, pois o extrato do piloto v0.5 não contém `VD3004` e não contém o módulo TIC.

In [ ]:
import shutil
if shutil.which('Rscript') is None:
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','-qq',
        'r-base','libcurl4-openssl-dev','libssl-dev','libxml2-dev'], check=True)
subprocess.run(['Rscript','-e',
    "options(repos=c(CRAN='https://cloud.r-project.org')); "
    "install.packages(c('PNADcIBGE','survey'), Ncpus=2)"], check=True)
print('R e pacotes instalados.')

### 2A — Reextrair a PNAD trimestral 2026T2 com nível de instrução `VD3004`

A nova extração preserva `VD3004` e `VD3005` separadamente. `VD3004` contém os níveis de instrução de 1 a 7; **a categoria TSE “lê e escreve” não pode ser obtida diretamente com `VD3004`**.

In [ ]:
subprocess.run(['Rscript', 'reextrair_pnad_2026T2_vd3004.R'],check=True)
arquivo = BASE / 'saida_fontes_v0_6/pnad_doadores_vd3004_2026T2.csv.gz'
assert arquivo.exists() and arquivo.stat().st_size>100000
print('Extraído:', arquivo.name, 'bytes:', arquivo.stat().st_size)
display(pd.read_csv(arquivo,nrows=6))

### 2B — Extrair a PNAD TIC 2025T4

Os campos `S07001` (utilização da Internet nos últimos três meses) e `S07006` (posse de telefone celular pessoal) registram **comportamentos diferentes**. Ter acesso à internet ou celular **não garante exposição a uma pesquisa digital nem disposição para responder**.

Esta extração usa `topic=4` (acumulado do quarto trimestre de 2025), não os microdados trimestrais básicos de 2026.

In [ ]:
subprocess.run(['Rscript', 'extrair_tic_2025T4.R'],check=True)
arquivo = BASE / 'saida_fontes_v0_6/pnad_tic_2025T4_16mais.csv.gz'
assert arquivo.exists() and arquivo.stat().st_size>100000
print('Extraído:', arquivo.name, 'bytes:', arquivo.stat().st_size)
display(pd.read_csv(arquivo,nrows=6))

## 3. Gerar o piloto enriquecido e validar

O programa usa, sucessivamente, doadores **ponderados** da PNAD 2026 para renda/ocupação/escolaridade, e doadores **ponderados distintos** da PNAD TIC 2025 para gerar conjuntamente dois indicadores digitais. Fallbacks demográficos são registrados individualmente. Não existe pareamento individual entre as duas pesquisas do IBGE.

O resultado permanecerá **provisório** até examinarmos as distribuições e os fallbacks reais.

In [ ]:
subprocess.run([
    sys.executable, 'gerar_piloto_v06.py',
    '--piloto', 'piloto_demografico_tse_50mil.csv',
    '--pnad2026', 'saida_fontes_v0_6/pnad_doadores_vd3004_2026T2.csv.gz',
    '--tic2025', 'saida_fontes_v0_6/pnad_tic_2025T4_16mais.csv.gz',
    '--saida', 'resultados_v0_6'
], check=True)
piloto = pd.read_csv('resultados_v0_6/piloto_socioeconomico_tic_50mil_v0_6.csv.gz')
assert len(piloto) == 50000
print('Piloto:', len(piloto), 'registros')
display(piloto.head())
display(pd.read_csv('resultados_v0_6/diagnostico_tic_fonte_ponderada_v0_6.csv').head())

## 4. Exportar resultados e extratos de auditoria

O ZIP de saída contém os dois arquivos PNAD extraídos, o piloto sintético e o manifesto. **Não contém o arquivo ZIP bruto original do IBGE**; a origem oficial fica indicada pelas URLs e pelo manifesto. Compartilhe o ZIP aqui na conversa para que possamos validar os valores reais da v0.6, especialmente diferenças entre a PNAD observada e o piloto imputado, a raridade de doadores e a ambiguidade de escolaridade.

In [ ]:
import zipfile
PACOTE=BASE/'auditoria_pnad_v0_6_resultados.zip'
with zipfile.ZipFile(PACOTE, 'w', zipfile.ZIP_DEFLATED) as z:
    for root in ['saida_fontes_v0_6','resultados_v0_6']:
        for p in (BASE/root).rglob('*'):
            if p.is_file() and 'downloads_' not in str(p):
                z.write(p, p.relative_to(BASE))
print('Pacote de resultados:', PACOTE, PACOTE.stat().st_size, 'bytes')
try:
    from google.colab import files
    files.download(str(PACOTE))
except ImportError:
    print('Em ambiente local, o arquivo está em:', PACOTE)